# Qwen3-32B bring-up — load the model + capture one example activation

**Purpose (M1 / unblock Lion).** Load Qwen3-32B 4-bit (nf4, Blackwell-verified) through the *production* `HFCapturer` and capture **one** residual-stream activation at **layer 32**, then save it as a tiny bundle Lion can build the Assistant-Axis probe against. Using `HFCapturer` (not ad-hoc code) means this example is byte-for-byte the same shape/dtype/runtime the full M4 run will emit.

What Lion gets out of this: a real `L32` activation `(1, 5120)` float16, plus the exact projection-relevant facts (layer index, token position, model id, compute precision).

**Kernel:** `pinchguard` (`uv run python -m ipykernel install --user --name pinchguard`).

> Scope: this is the bring-up / reference notebook — a *single* turn, no scenario, no loop. The 15-turn agent run is a separate notebook (M4).

## 1 · Config & environment

Loads `.env` (HF_TOKEN, HF_HOME) **before** importing transformers, puts the repo root on `sys.path` so `shim.capture` imports, and pins the bring-up knobs. The model path points at the completed local download (all 17 shards present).

In [1]:
import os
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up until we find pyproject.toml so the notebook is run-location agnostic."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("could not locate repo root (no pyproject.toml above this notebook)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def _load_dotenv(path: Path) -> None:
    """Minimal .env loader (avoids a python-dotenv dep). Sets HF_TOKEN / HF_HOME
    so the HF cache resolves to the lab mount, not ~/.cache."""
    if not path.exists():
        print(f"[config] no .env at {path} — relying on inherited environment")
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        os.environ.setdefault(key.strip(), val.strip())


_load_dotenv(REPO_ROOT / ".env")

# --- Bring-up knobs ---------------------------------------------------------
# Completed local download: /datapool/.../models holds all 17 shards + tokenizer
# + config.json directly (HF_HOME root). from_pretrained loads it by path and
# ignores the sibling hub/ and xet/ dirs.
MODEL_PATH = os.environ.get(
    "MODEL_NAME", "/datapool/analysis_data/tara/pinchguard/models"
)
LAYER = 32  # confirmed: num_hidden_layers=64 -> midpoint 32; same layer the
            # safety-research/assistant-axis probe extracts the axis at.
TOKEN_POSITION = "last_input"  # the only position HFCapturer implements today.
QUANT = "nf4"                  # Q4: verified end-to-end on Blackwell (sm_120).
# DEVICE_MAP is chosen in the GPU-sanity cell below — a SINGLE GPU.
# IMPORTANT: do NOT use device_map="auto" here. Splitting a bitsandbytes
# 4-bit model across both GPUs corrupts the forward pass on this box
# (verified): generation AND the captured activations come out garbage.
# 4-bit Qwen3-32B is ~19 GB and fits on one 24 GB card.
MAX_NEW_TOKENS = 512          # Qwen3 thinks first (~150 tokens here) THEN answers
                              # after </think>; 128 truncated mid-thought so the
                              # final response never appeared. 512 leaves headroom
                              # to close </think> and emit the answer. (The captured
                              # activation is independent of generation length — it
                              # comes from the prompt-only forward pass.)

# Where the example bundle for Lion lands. Committable artefacts stay tiny (one
# npz + a README); the heavy weights never leave the mount.
OUT_DIR = REPO_ROOT / "notebooks" / "example_activation"

print(f"repo root : {REPO_ROOT}")
print(f"model     : {MODEL_PATH}")
print(f"HF_HOME   : {os.environ.get('HF_HOME', '<unset>')}")
print(f"layer     : {LAYER}  token_position: {TOKEN_POSITION}  quant: {QUANT}")
assert Path(MODEL_PATH).joinpath("config.json").exists(), f"no config.json under {MODEL_PATH}"

repo root : /home/geoff/dev/pinchguard
model     : /datapool/analysis_data/tara/pinchguard/models
HF_HOME   : /datapool/analysis_data/tara/pinchguard/models
layer     : 32  token_position: last_input  quant: nf4


## 2 · GPU + model-config sanity

Confirm CUDA is live, the GPUs have headroom, and the on-disk config matches the numbers the probe relies on (`num_hidden_layers=64`, `hidden_size=5120`).

In [2]:
import json

import torch

assert torch.cuda.is_available(), "CUDA not available — HFCapturer 4-bit needs a GPU"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    name = torch.cuda.get_device_name(i)
    print(f"cuda:{i}  {name}  free {free/1e9:.1f} GB / {total/1e9:.1f} GB")

cfg = json.loads((Path(MODEL_PATH) / "config.json").read_text())
print("\nmodel_type        :", cfg.get("model_type"))
print("num_hidden_layers :", cfg.get("num_hidden_layers"))
print("hidden_size       :", cfg.get("hidden_size"))
assert 0 <= LAYER < cfg["num_hidden_layers"], "LAYER out of range for this model"
HIDDEN_SIZE = cfg["hidden_size"]

# Pick the single GPU with the most free memory (avoid device_map='auto';
# see the config cell — multi-GPU 4-bit split garbages the forward pass).
_free = {i: torch.cuda.mem_get_info(i)[0] for i in range(torch.cuda.device_count())}
_best = max(_free, key=_free.get)
DEVICE_MAP = f"cuda:{_best}"
print(f"\nselected DEVICE_MAP = {DEVICE_MAP}  (free {_free[_best]/1e9:.1f} GB)")

cuda:0  NVIDIA RTX PRO 4000 Blackwell  free 17.2 GB / 25.2 GB
cuda:1  NVIDIA RTX PRO 4000 Blackwell  free 24.9 GB / 25.2 GB

model_type        : qwen3
num_hidden_layers : 64
hidden_size       : 5120

selected DEVICE_MAP = cuda:1  (free 24.9 GB)


## 3 · Load Qwen3-32B (4-bit) through the production `HFCapturer`

This is the same class the shim uses in production. It loads the model under a bitsandbytes nf4 config (double-quant, fp16 compute), forward-hooks `model.model.layers[32]`, and exposes `.capture(messages, step_id)`.

Loads onto a **single GPU** (~19 GB resident, ~10–20 s). We pin one device on purpose: `device_map="auto"` splits the 4-bit weights across both GPUs and **corrupts the forward pass** here — coherent text and valid activations require the model on one device.

In [3]:
import time

from shim.capture import HFCapturer

t0 = time.time()
capturer = HFCapturer(
    model_name=MODEL_PATH,
    layers=(LAYER,),
    token_position=TOKEN_POSITION,
    max_new_tokens=MAX_NEW_TOKENS,
    device_map=DEVICE_MAP,
    quantization=QUANT,
)
print(f"loaded in {time.time() - t0:.1f}s")
print("model_id:", capturer.model_id)
print("decoder layers:", len(capturer._layer_modules))
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} resident ~{(total-free)/1e9:.1f} GB")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

/home/geoff/dev/pinchguard/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


loaded in 11.4s
model_id: /datapool/analysis_data/tara/pinchguard/models
decoder layers: 64
cuda:0 resident ~7.9 GB
cuda:1 resident ~20.4 GB


## 4 · Capture one example activation

A deliberately **neutral** default-Assistant prompt (matching the probe's default condition — no character/persona). The activation we keep is the `last_input` hidden state at layer 32 from the prompt-only forward pass; generation is separate and doesn't bleed into it.

In [4]:
import uuid

messages = [
    {"role": "system", "content": "You are a helpful, honest, careful assistant."},
    {"role": "user", "content": "In one sentence, what is the capital of France?"},
]
step_id = str(uuid.uuid4())

t0 = time.time()
result = capturer.capture(messages, step_id)
print(f"captured + generated in {time.time() - t0:.1f}s\n")

act = result.activations[f"L{LAYER}"]
print("activation key   :", f"L{LAYER}")
print("activation shape :", act.shape)
print("activation dtype :", act.dtype)
print("activation_meta  :", result.activation_meta)

# Invariants Lion's projection depends on.
assert act.shape == (1, HIDDEN_SIZE), f"expected (1, {HIDDEN_SIZE}), got {act.shape}"
assert str(act.dtype) == "float16", f"expected float16, got {act.dtype}"
assert result.activation_meta["capture_runtime"] == "hf-eager", "not real weights!"
assert result.activation_meta["layers"] == [LAYER]
print("\nall invariants hold ✔")

/home/geoff/dev/pinchguard/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


captured + generated in 9.1s

activation key   : L32
activation shape : (1, 5120)
activation dtype : float16
activation_meta  : {'token_position': 'last_input', 'layers': [32], 'dtype': 'float16', 'capture_runtime': 'hf-eager', 'quantization': 'nf4'}

all invariants hold ✔


## 5 · Inspect the completion (thinking on)

Qwen3's chat template enables thinking by default, so the raw completion carries a `<think>…</think>` block **followed by the final answer** (both land in `output_raw` in the real run). With `MAX_NEW_TOKENS=512` the model finishes thinking, closes `</think>`, and emits the response — at 128 the thinking alone exhausted the budget and the answer never appeared.

In [5]:
print("--- prompt (rendered) ---")
print(result.prompt_text[-400:])
print("\n--- completion (raw) ---")
print(result.completion_text)

# The completion must close </think> AND carry a final answer after it, or the
# generation budget truncated the reasoning (see cell 1: MAX_NEW_TOKENS).
has_open = "<think>" in result.completion_text
has_close = "</think>" in result.completion_text
final_answer = (
    result.completion_text.split("</think>")[-1].strip() if has_close else ""
)
print("\n<think> present     :", has_open)
print("</think> closed      :", has_close)
print("final answer present :", bool(final_answer))
print("final answer         :", repr(final_answer))
assert has_open and has_close, "thinking block was truncated — raise MAX_NEW_TOKENS"
assert final_answer, "no response after </think> — raise MAX_NEW_TOKENS"

--- prompt (rendered) ---
<|im_start|>system
You are a helpful, honest, careful assistant.<|im_end|>
<|im_start|>user
In one sentence, what is the capital of France?<|im_end|>
<|im_start|>assistant


--- completion (raw) ---
<think>
Okay, the user is asking for the capital of France in one sentence. Let me make sure I remember correctly. France is a country in Europe, and I think the capital is Paris. Wait, is there any chance I'm mixing it up with another country? No, I'm pretty sure Paris is correct. Let me double-check. Yeah, Paris has been the capital for a long time. I don't think there's any recent change. So the answer should be straightforward. Just need to state it clearly in one sentence without any extra fluff. Maybe something like "The capital of France is Paris." That's concise and accurate. Alright, ready to respond.
</think>

The capital of France is Paris.

<think> present     : True
</think> closed      : True
final answer present : True
final answer         : 'The cap

## 6 · Save the example bundle for Lion

Writes a tiny, committable bundle: the `L32` activation as `.npz` (same `np.savez_compressed` format the shim writes per step) plus a `README.md` manifest with the projection facts. This is the reference Lion checks his Assistant-Axis extraction against.

In [6]:
import numpy as np

OUT_DIR.mkdir(parents=True, exist_ok=True)
npz_path = OUT_DIR / f"{step_id}.npz"
np.savez_compressed(npz_path, **result.activations)

manifest = {
    "model_id": capturer.model_id,
    "layer": LAYER,
    "hook": f"module-output of model.model.layers[{LAYER}]",
    "token_position": TOKEN_POSITION,
    "hidden_dim": HIDDEN_SIZE,
    "activation_key": f"L{LAYER}",
    "dtype": str(act.dtype),
    "quantization": QUANT,
    "capture_runtime": result.activation_meta["capture_runtime"],
    "step_id": step_id,
    "npz": npz_path.name,
}

readme = "\n".join(
    [
        "# Example activation bundle (Qwen3-32B, layer 32)",
        "",
        "One residual-stream activation captured via the production `HFCapturer`,",
        "for validating the Assistant-Axis probe before the full M4 run exists.",
        "",
        "## Facts",
        *[f"- **{k}**: `{v}`" for k, v in manifest.items()],
        "",
        "## Load it",
        "```python",
        "import numpy as np",
        f'a = np.load("{npz_path.name}")["L{LAYER}"]  # shape (1, {HIDDEN_SIZE}), float16',
        "```",
        "",
        "## Open cross-check with Lion",
        f"`L{LAYER}` here is the **module-output of decoder block index {LAYER}**",
        f"(`model.model.layers[{LAYER}]`). Confirm the probe extracts the axis at the",
        f"same definition (vs `hidden_states[{LAYER}]`, which is the *input* to block",
        f"{LAYER} / output of block {LAYER - 1}) before trusting a projection.",
    ]
)
(OUT_DIR / "README.md").write_text(readme + "\n")
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")

print("wrote bundle to:", OUT_DIR)
for p in sorted(OUT_DIR.iterdir()):
    print("  ", p.name, f"({p.stat().st_size} bytes)")
print("\nmanifest:")
print(json.dumps(manifest, indent=2))

wrote bundle to: /home/geoff/dev/pinchguard/notebooks/example_activation
   3aa8491e-bb42-408f-a549-2657361a304d.npz (8131 bytes)
   README.md (1047 bytes)
   manifest.json (404 bytes)

manifest:
{
  "model_id": "/datapool/analysis_data/tara/pinchguard/models",
  "layer": 32,
  "hook": "module-output of model.model.layers[32]",
  "token_position": "last_input",
  "hidden_dim": 5120,
  "activation_key": "L32",
  "dtype": "float16",
  "quantization": "nf4",
  "capture_runtime": "hf-eager",
  "step_id": "3aa8491e-bb42-408f-a549-2657361a304d",
  "npz": "3aa8491e-bb42-408f-a549-2657361a304d.npz"
}
